# Olist Delivery Delay Prediction
### Brazilian E-Commerce Public Dataset | Capstone Project
#### Notebook 02 — Data Merging

---

**Authors:** Sura and Aman

**Verified by:** Ameed and Ruaa


## Goal

Combine all cleaned tables into a single dataset where one row = one order.

This dataset feeds directly into Notebook 03 (EDA) and Notebook 04 (Modeling).

**Design Principles:**
- Preserve row count during joins (anchor = orders table)
- Avoid data leakage
- Maintain one-row-per-order structure throughout
- Keep modeling data separate from analysis-heavy data (e.g. full geolocation)


---
## Load Cleaned Tables

All datasets were cleaned and saved in Notebook 01. We reload them here with proper date parsing.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PATH = '../data/processed/'

orders      = pd.read_csv(PATH + 'orders_clean.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
items       = pd.read_csv(PATH + 'items_clean.csv', parse_dates=['shipping_limit_date'])
payments    = pd.read_csv(PATH + 'payments_clean.csv')
reviews     = pd.read_csv(PATH + 'reviews_clean.csv')
customers   = pd.read_csv(PATH + 'customers_clean.csv')
sellers     = pd.read_csv(PATH + 'sellers_clean.csv')
products    = pd.read_csv(PATH + 'products_clean.csv')
geolocation = pd.read_csv(PATH + 'geolocation_clean.csv')
translation = pd.read_csv(PATH + 'translation_clean.csv')

tables = {
    'orders': orders, 'items': items, 'payments': payments,
    'reviews': reviews, 'customers': customers, 'sellers': sellers,
    'products': products, 'geolocation': geolocation, 'translation': translation
}

for name, df in tables.items():
    print(f'{name:<15} {df.shape[0]:>10,} rows | {df.shape[1]:>2} columns')


orders              96,455 rows | 13 columns
items              112,650 rows |  9 columns
payments           103,886 rows |  5 columns
reviews             99,224 rows |  7 columns
customers           99,441 rows |  6 columns
sellers              3,095 rows |  5 columns
products            32,949 rows |  9 columns
geolocation      1,000,163 rows |  7 columns
translation             71 rows |  2 columns


---
## Geolocation Strategy

The geolocation dataset contains ~1M rows — multiple coordinate entries per zip code.

**Problem:** A direct join would create duplicates and break the one-row-per-order structure.

**Solution:** Aggregate to one coordinate per zip code using the mean (centroid approximation).

**Design Decision:** The aggregated version is used for modeling joins. The full table is
preserved in `geolocation_clean.csv` for spatial density analysis in EDA.

In [2]:
# ── Aggregate geolocation to one coordinate per zip code ─────────────────────
geo_lookup = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(lat=('geolocation_lat', 'mean'), lng=('geolocation_lng', 'mean'))
)

customer_geo = geo_lookup.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'lat': 'customer_lat', 'lng': 'customer_lng'
})

seller_geo = geo_lookup.rename(columns={
    'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
    'lat': 'seller_lat', 'lng': 'seller_lng'
})

print(f'Reduced ~1M rows → {len(geo_lookup):,} unique zip codes')
print('One-to-one join compatibility achieved.')


Reduced ~1M rows → 19,015 unique zip codes
One-to-one join compatibility achieved.


---
## Prepare Items

Items table has one row per item — must be aggregated to order level.

### Fix Missing Translations

Two product categories exist in the products table but are missing from the translation table.
We detect them programmatically and add manual translations.

In [3]:
# ── Detect missing translations ───────────────────────────────────────────────
missing_categories = (
    set(products['product_category_name'].dropna())
    - set(translation['product_category_name'])
)
missing_categories.discard('unknown')
print('Missing categories:', missing_categories)


Missing categories: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


In [4]:
# ── Apply translation patch ───────────────────────────────────────────────────
missing_cats = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_food_preparators',
}

for pt, en in missing_cats.items():
    if pt not in translation['product_category_name'].values:
        translation = pd.concat([
            translation,
            pd.DataFrame([{'product_category_name': pt,
                           'product_category_name_english': en}])
        ], ignore_index=True)

print(f'Translation table size: {len(translation)} categories')
print('All real categories now covered.')


Translation table size: 73 categories
All real categories now covered.


### Enrich and Aggregate Items to Order Level

In [5]:
# ── Enrich items and aggregate to order level ─────────────────────────────────
items_enriched = (
    items
    .merge(products, on='product_id', how='left')
    .merge(translation, on='product_category_name', how='left')
    .merge(sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state', 'seller_city']],
           on='seller_id', how='left')
)

items_enriched['product_volume_cm3'] = (
    items_enriched['product_length_cm'] *
    items_enriched['product_height_cm'] *
    items_enriched['product_width_cm']
)

items_agg = (
    items_enriched
    .groupby('order_id', as_index=False)
    .agg(
        n_items=('order_item_id', 'max'),
        n_unique_sellers=('seller_id', 'nunique'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        avg_product_weight_g=('product_weight_g', 'mean'),
        avg_product_volume_cm3=('product_volume_cm3', 'mean'),
        product_category=('product_category_name_english',
                          lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
        seller_zip_code_prefix=('seller_zip_code_prefix', 'first'),
        seller_state=('seller_state', 'first')
    )
)

print(f'Items aggregated: {len(items_enriched):,} rows → {len(items_agg):,} orders')


Items aggregated: 112,650 rows → 98,666 orders


---
## Prepare Payments

Multiple payment rows per order → aggregate to a single row per order.

In [6]:
# ── Aggregate payments to order level ────────────────────────────────────────
payments_agg = (
    payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value=('payment_value', 'sum'),
        n_payment_methods=('payment_sequential', 'max'),
        max_installments=('payment_installments', 'max'),
        payment_type=('payment_type',
                      lambda x: x.mode()[0] if not x.mode().empty else 'unknown')
    )
)

print(f'Payments aggregated: {len(payments):,} rows → {len(payments_agg):,} orders')


Payments aggregated: 103,886 rows → 99,440 orders


---
## Prepare Reviews

We keep only `review_score` and ensure one row per order.
Orders with multiple reviews: we keep the first review as a simplification.

In [7]:
# ── Deduplicate reviews ────────────────────────────────────────────────────────
reviews_slim = (
    reviews
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'review_score']]
)

print(f'Reviews deduplicated: {len(reviews):,} rows → {len(reviews_slim):,} orders')


Reviews deduplicated: 99,224 rows → 98,673 orders


---
## Build Final Dataset

We merge all tables using LEFT JOINs anchored to the orders table.
This ensures row count is preserved — the orders table drives the final row count.

In [8]:
# ── Merge all tables ──────────────────────────────────────────────────────────
df = orders.copy()

df = df.merge(customers[['customer_id', 'customer_unique_id',
                          'customer_zip_code_prefix', 'customer_state', 'customer_city']],
              on='customer_id', how='left')

df = df.merge(items_agg,    on='order_id', how='left')
df = df.merge(payments_agg, on='order_id', how='left')
df = df.merge(reviews_slim, on='order_id', how='left')
df = df.merge(customer_geo, on='customer_zip_code_prefix', how='left')
df = df.merge(seller_geo,   on='seller_zip_code_prefix',   how='left')

print(f'Final shape: {df.shape}')
print(f'Row count preserved: {len(df):,} (matches orders: {len(orders):,})')


Final shape: (96455, 35)
Row count preserved: 96,455 (matches orders: 96,455)


---
### Quality Checks

In [9]:
# ── Quality checks ────────────────────────────────────────────────────────────
print(f'Rows         : {len(df):,}')
print(f'Unique orders: {df["order_id"].nunique():,}')
print(f'Duplicates   : {df["order_id"].duplicated().sum()}')
print(f'Late rate    : {df["is_late"].mean()*100:.2f}%')
print(f'\nMissing values (non-zero only):')
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False).to_string())


Rows         : 96,455
Unique orders: 96,455
Duplicates   : 0
Late rate    : 8.11%

Missing values (non-zero only):
review_score              646
customer_lat              264
customer_lng              264
seller_lat                215
seller_lng                215
avg_product_weight_g       16
avg_product_volume_cm3     16
total_payment_value         1
n_payment_methods           1
max_installments            1
payment_type                1


---
### Handle Remaining Missing Values

In [10]:
# ── Fill remaining missing values ────────────────────────────────────────────
# review_score: keep as NaN — absence of review is informative (has_review flag handles it)
df['has_review'] = df['review_score'].notna().astype(int)

# product_category: orders without item data
df['product_category'] = df['product_category'].fillna('unknown')

# product dimensions: fill with median (robust to skew)
df['avg_product_weight_g']   = df['avg_product_weight_g'].fillna(df['avg_product_weight_g'].median())
df['avg_product_volume_cm3'] = df['avg_product_volume_cm3'].fillna(df['avg_product_volume_cm3'].median())

# payment fields: orders without payment data (edge cases)
df['n_payment_methods']   = df['n_payment_methods'].fillna(0).astype(int)
df['max_installments']    = df['max_installments'].fillna(0).astype(int)
df['total_payment_value'] = df['total_payment_value'].fillna(0)
df['payment_type']        = df['payment_type'].fillna('unknown')

print('Missing values handled.')
print(f'\nRemaining missing values (expected — geographic only):')
remaining = df.isnull().sum()
print(remaining[remaining > 0].sort_values(ascending=False).to_string())
print('\nNote: review_score and lat/lng missing values are intentional.')
print('  review_score: captured by has_review flag')
print('  lat/lng: customer_state used as geographic feature instead')


Missing values handled.

Remaining missing values (expected — geographic only):
review_score    646
customer_lat    264
customer_lng    264
seller_lat      215
seller_lng      215

Note: review_score and lat/lng missing values are intentional.
  review_score: captured by has_review flag
  lat/lng: customer_state used as geographic feature instead


---
### Save Final Dataset

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
df.to_csv('../data/processed/olist_merged.csv', index=False)

print(f'Saved: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Late rate: {df["is_late"].mean()*100:.2f}%')


Saved: (96455, 36)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_late', 'is_delivered', 'delivery_status', 'delivery_days', 'has_timestamp_issue', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_state', 'customer_city', 'n_items', 'n_unique_sellers', 'total_price', 'total_freight', 'avg_product_weight_g', 'avg_product_volume_cm3', 'product_category', 'seller_zip_code_prefix', 'seller_state', 'total_payment_value', 'n_payment_methods', 'max_installments', 'payment_type', 'review_score', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'has_review']
Late rate: 8.11%


### Final Result

- **96,455 rows** — one row per order, row count preserved throughout all joins
- **36 columns** — features from all 9 source tables, aggregated to order level
- **8.1% late rate** — consistent with Notebook 01
- Ready for EDA in Notebook 03

**Missing values in final dataset (intentional):**

| Column | Missing | Reason |
|--------|---------|--------|
| `review_score` | 646 (0.7%) | Captured by `has_review` flag; dropped in modeling due to leakage |
| `customer_lat/lng` | 264 (0.27%) | `customer_state` used as geographic feature instead |
| `seller_lat/lng` | 215 (0.22%) | `seller_state` used instead |

---
## Final Note on Geolocation

Geolocation data (~1M rows) was aggregated to one coordinate per zip code using the mean
(centroid approximation) to preserve the one-row-per-order structure.

The full geolocation table (`geolocation_clean.csv`) is preserved separately for
spatial density analysis in EDA.

> **Next step:** `03_eda.ipynb` → Exploratory Data Analysis


## Final Dataset Schema

Each row represents one delivered order with aggregated features from all source tables.

| Column | Source | Type | Notes |
|--------|--------|------|-------|
| order_id | orders | string | Primary key |
| customer_id | orders | string | Per-order customer key |
| order_status | orders | string | Always 'delivered' after scope filter |
| order_purchase_timestamp | orders | datetime | Purchase time |
| order_approved_at | orders | datetime | Payment approval |
| order_delivered_carrier_date | orders | datetime | Carrier handoff |
| order_delivered_customer_date | orders | datetime | Actual delivery |
| order_estimated_delivery_date | orders | datetime | Promised delivery |
| **is_late** | orders | int (0/1) | **Target variable** |
| is_delivered | orders | int (0/1) | Always 1 after scope filter |
| delivery_status | orders | string | 'early' or 'late' |
| has_timestamp_issue | orders | int (0/1) | 1 = suspicious timestamp ordering |
| delivery_days | orders | int | Days from purchase to delivery |
| customer_unique_id | customers | string | True customer ID |
| customer_zip_code_prefix | customers | int | Zip code prefix |
| customer_state | customers | string | State abbreviation |
| customer_city | customers | string | Customer city |
| n_items | items | int | Number of items in order |
| n_unique_sellers | items | int | Number of distinct sellers |
| total_price | items | float | Sum of item prices |
| total_freight | items | float | Sum of freight costs |
| avg_product_weight_g | items | float | Mean product weight |
| avg_product_volume_cm3 | items | float | Mean product volume |
| product_category | items | string | Most common category in order |
| seller_zip_code_prefix | items | int | First seller zip code |
| seller_state | items | string | First seller state |
| total_payment_value | payments | float | Total amount paid |
| n_payment_methods | payments | int | Number of payment methods used |
| max_installments | payments | int | Maximum installments |
| payment_type | payments | string | Most common payment method |
| review_score | reviews | float | 1–5 rating (NaN if no review) |
| has_review | reviews | int (0/1) | 1 = customer submitted a review |
| customer_lat | geolocation | float | Customer latitude |
| customer_lng | geolocation | float | Customer longitude |
| seller_lat | geolocation | float | Seller latitude |
| seller_lng | geolocation | float | Seller longitude |
